# Streaming SSE Server — LangGraph with astream_events
## Push Tokens as They Arrive — SSE and astream_events
⏱ ~15 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/81-streaming-sse-server/streaming_sse_server_workbook.ipynb)

Demonstrates a think→answer LangGraph graph with token-level streaming via astream_events. Shows the FastAPI + EventSourceResponse pattern as reference code. The streaming graph itself runs in Colab.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — SSE vs polling; when streaming UX matters |
| 2 | **Streaming Graph** — think_node → answer_node; both nodes use streaming ChatOpenAI |
| 3 | **astream_events v1** — Filter on_chat_model_stream to print tokens as they arrive |
| 4 | **Batch vs Stream** — Compare output latency: invoke vs astream_events |
| 5 | **FastAPI Reference** — Full SSE server pattern (reference only) |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

## Part 1 — Streaming Event Taxonomy

### Beyond token streaming

Example 75 covered `on_chat_model_stream` for token-by-token output. A production SSE server also needs to stream:
- **Tool call events** — when the agent decides to call a tool and when results come back
- **Node progress** — which graph node is currently running
- **Errors** — structured error events the client can render

### Full `astream_events` event types

| Event | When it fires | Useful payload |
|-------|-------------|----------------|
| `on_chain_start` | Graph invocation begins | `run_id` |
| `on_node_start` | A graph node starts | `name` (node name) |
| `on_chat_model_stream` | One LLM token arrives | `data.chunk.content` |
| `on_tool_start` | A tool call begins | `name`, `data.input` |
| `on_tool_end` | A tool call completes | `data.output` |
| `on_chain_end` | Graph invocation completes | full final state |

A frontend that only handles `token` events will miss tool activity entirely — the screen looks frozen while tools run.

### SSE event types for the client

Map LangGraph events to named SSE events:

```
on_node_start      → event: "node"   data: "think_node"
on_tool_start      → event: "tool"   data: "search: {query}"
on_chat_model_stream → event: "token" data: "The "
on_chain_end       → event: "done"   data: "[DONE]"
```

> **Prerequisite:** Familiar with `astream_events` basics (example 75).

In [ ]:
import asyncio, time
from typing import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode

# define a simple tool the agent can call
@tool
def calculator(expression: str) -> str:
    '''Evaluate a mathematical expression. Input: a Python math expression string.'''
    try:
        result = eval(expression, {"__builtins__": {}}, {"abs": abs, "round": round})
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error: {e}"

@tool
def word_count(text: str) -> str:
    '''Count words in a text string.'''
    return f"Word count: {len(text.split())}"

tools = [calculator, word_count]
llm_with_tools = ChatOpenAI(model="gpt-5.4-nano", temperature=0).bind_tools(tools)

class AgentState(TypedDict):
    question: str
    messages: list

def agent_node(state: AgentState) -> dict:
    msgs = state["messages"] + [HumanMessage(content=state["question"])]
    r = llm_with_tools.invoke(msgs)
    return {"messages": msgs + [r]}

def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    return "tools" if getattr(last, "tool_calls", None) else END

g = StateGraph(AgentState)
g.add_node("agent",  agent_node)
g.add_node("tools",  ToolNode(tools))
g.add_edge(START, "agent")
g.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
g.add_edge("tools", "agent")
tool_app = g.compile()

# list all event types for a tool-using invocation
async def list_events(question: str):
    print(f"Events for: {question!r}\n")
    async for event in tool_app.astream_events(
        {"question": question, "messages": []}, version="v1"
    ):
        name = event["event"]
        if name in ("on_node_start", "on_tool_start", "on_tool_end",
                    "on_chat_model_stream", "on_chain_end"):
            if name == "on_chat_model_stream":
                preview = repr(event["data"]["chunk"].content[:20])
            elif name == "on_tool_start":
                preview = f"tool={event['name']!r} input={str(event['data']['input'])[:40]}"
            elif name == "on_tool_end":
                preview = f"output={str(event['data']['output'])[:40]}"
            else:
                preview = event.get("name", "")
            print(f"  {name:<30} {preview}")

await list_events("What is 15 * 23? Also count words in 'hello world foo'.")

## Part 2 — Multi-Turn Streaming with Memory

Multi-turn conversations require the graph to carry message history across invocations. The pattern:

1. Store the full message list in state (a `list[BaseMessage]`)
2. Each invocation appends to the list and passes it back to the agent
3. The agent sends the full history to the LLM on each call

### Thread IDs and checkpointing

For true multi-turn persistence (across process restarts or HTTP requests), use a `MemorySaver` checkpointer and a `thread_id`:

```python
cfg = {"configurable": {"thread_id": "user-session-123"}}
app.invoke({"messages": [HumanMessage("Hello")]}, config=cfg)
# Later, in a new HTTP request:
app.invoke({"messages": [HumanMessage("What did I just say?")]}, config=cfg)
# The graph restores the prior conversation from the checkpoint
```

Without a checkpointer, you must pass the full history in the state on every call.

In [ ]:
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import MemorySaver

class ConversationState(TypedDict):
    messages: list

def conversation_agent(state: ConversationState) -> dict:
    r = llm_with_tools.invoke(state["messages"])
    return {"messages": state["messages"] + [r]}

def conv_should_continue(state: ConversationState) -> str:
    last = state["messages"][-1]
    return "tools" if getattr(last, "tool_calls", None) else END

gc = StateGraph(ConversationState)
gc.add_node("agent", conversation_agent)
gc.add_node("tools", ToolNode(tools))
gc.add_edge(START, "agent")
gc.add_conditional_edges("agent", conv_should_continue, {"tools": "tools", END: END})
gc.add_edge("tools", "agent")
ck = MemorySaver()
conv_app = gc.compile(checkpointer=ck)

# multi-turn conversation: each message references the prior context
CONVERSATION = [
    "What is 7 * 8?",
    "Now double that result.",
    "What calculations have you done so far?",
]

thread_cfg = {"configurable": {"thread_id": "demo-conversation-1"}}

async def stream_turn(question: str, cfg: dict):
    print(f"\nUser: {question}")
    print("Agent: ", end="", flush=True)
    current_msgs = conv_app.get_state(cfg).values.get("messages", [])
    new_msgs = current_msgs + [HumanMessage(content=question)]

    async for event in conv_app.astream_events(
        {"messages": new_msgs}, version="v1", config=cfg
    ):
        if event["event"] == "on_chat_model_stream":
            chunk = event["data"]["chunk"].content
            if chunk:
                print(chunk, end="", flush=True)
    print()

async def run_conversation():
    for turn in CONVERSATION:
        await stream_turn(turn, thread_cfg)

await run_conversation()

## Part 3 — Production FastAPI SSE Server

A complete production SSE server needs to handle:
1. **Multiple event types** — token, node progress, tool activity, done/error
2. **CORS** — so browser frontends on different ports can connect
3. **Error handling** — catch exceptions and yield a structured `error` event
4. **Connection cleanup** — the `EventSourceResponse` generator handles this automatically when the client disconnects

### Reference implementation

> **Note:** This cell is reference code only — run locally with `uvicorn`, not in Colab.

```bash
pip install fastapi sse-starlette uvicorn
uvicorn main:api --host 0.0.0.0 --port 8000
```

Test with:
```bash
curl -N "http://localhost:8000/stream?question=What+is+7+times+8"
```

In [ ]:
# FastAPI SSE Server — reference implementation (run locally, not in Colab)
FASTAPI_SERVER = '''
import asyncio
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from sse_starlette.sse import EventSourceResponse
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict

# --- build graph (same as notebook) ---
@tool
def calculator(expression: str) -> str:
    # evaluate a math expression safely
    try:
        return f"{expression} = {eval(expression, {}, {})}"
    except Exception as e:
        return f"Error: {e}"

tools = [calculator]
llm   = ChatOpenAI(model="gpt-5.4-nano", temperature=0).bind_tools(tools)

class State(TypedDict):
    messages: list

def agent(state):
    return {"messages": state["messages"] + [llm.invoke(state["messages"])]}

def should_continue(state):
    return "tools" if getattr(state["messages"][-1], "tool_calls", None) else END

g = StateGraph(State)
g.add_node("agent", agent); g.add_node("tools", ToolNode(tools))
g.add_edge(START, "agent")
g.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
g.add_edge("tools", "agent")
app = g.compile(checkpointer=MemorySaver())

# --- FastAPI ---
api = FastAPI()
api.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@api.get("/stream")
async def stream_endpoint(question: str, session_id: str = "default"):
    cfg = {"configurable": {"thread_id": session_id}}

    async def generate():
        try:
            msgs = app.get_state(cfg).values.get("messages", [])
            init = {"messages": msgs + [HumanMessage(content=question)]}

            async for event in app.astream_events(init, version="v1", config=cfg):
                ename = event["event"]
                if ename == "on_node_start":
                    yield {"event": "node",  "data": event["name"]}
                elif ename == "on_tool_start":
                    yield {"event": "tool",  "data": f"{event['name']}: {str(event['data']['input'])[:80]}"}
                elif ename == "on_chat_model_stream":
                    chunk = event["data"]["chunk"].content
                    if chunk:
                        yield {"event": "token", "data": chunk}
            yield {"event": "done", "data": "[DONE]"}
        except Exception as e:
            yield {"event": "error", "data": str(e)}

    return EventSourceResponse(generate())
'''

print("FastAPI SSE Server reference implementation:")
print(FASTAPI_SERVER)

## Part 4 — HTML/JS Frontend SSE Client

The frontend uses the browser's native `EventSource` API to receive SSE events. No library needed — it's built into every modern browser.

```javascript
const es = new EventSource('/stream?question=Hello&session_id=user123');

es.addEventListener('node',  e => showProgress(e.data));
es.addEventListener('tool',  e => showToolCall(e.data));
es.addEventListener('token', e => appendToken(e.data));
es.addEventListener('done',  e => { finalize(); es.close(); });
es.addEventListener('error', e => showError(e.data));
```

The frontend below is a complete, self-contained HTML file. Save it alongside the FastAPI server and open it in a browser with the server running on port 8000.

In [ ]:
# Full HTML/JS SSE client — save to sse_client.html and open with the FastAPI server running
HTML_CLIENT = '''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>LangGraph SSE Stream</title>
  <style>
    body { font-family: system-ui, sans-serif; max-width: 700px; margin: 40px auto; padding: 0 20px; }
    #output { font-family: monospace; white-space: pre-wrap; border: 1px solid #ddd;
              padding: 16px; border-radius: 8px; min-height: 120px; background: #fafafa; }
    .progress { color: #666; font-style: italic; }
    .tool     { color: #0066cc; }
    .done     { color: #009900; font-weight: bold; }
    .error    { color: #cc0000; }
    button    { padding: 8px 16px; cursor: pointer; }
    input     { width: 480px; padding: 8px; margin-right: 8px; }
  </style>
</head>
<body>
  <h2>LangGraph SSE Stream</h2>
  <div>
    <input id="question" type="text" value="What is 7 times 8?" placeholder="Ask a question...">
    <button onclick="startStream()">Ask</button>
  </div>
  <br>
  <div id="output"></div>

  <script>
    let es = null;
    const output = document.getElementById('output');
    const input  = document.getElementById('question');

    function startStream() {
      if (es) { es.close(); }
      output.innerHTML = '';

      const question   = input.value;
      const sessionId  = 'session-' + Date.now();
      const url        = `/stream?question=${encodeURIComponent(question)}&session_id=${sessionId}`;

      es = new EventSource(url);

      es.addEventListener('node', e => {
        output.innerHTML += `<span class="progress">[${e.data}] </span>`;
      });

      es.addEventListener('tool', e => {
        output.innerHTML += `<span class="tool">\n[tool: ${e.data}]\n</span>`;
      });

      es.addEventListener('token', e => {
        output.innerHTML += e.data;
      });

      es.addEventListener('done', e => {
        output.innerHTML += `<span class="done">\n\n[DONE]</span>`;
        es.close();
      });

      es.addEventListener('error', e => {
        output.innerHTML += `<span class="error">\n[Error: ${e.data}]</span>`;
        es.close();
      });
    }
  </script>
</body>
</html>'''

with open("/tmp/sse_client.html", "w") as f:
    f.write(HTML_CLIENT)

print("Saved to /tmp/sse_client.html")
print("Open this file in a browser with the FastAPI server running on port 8000.")
print()
print(HTML_CLIENT[:500], "...")

## Exercises

---

**Exercise 1 — Add a `thinking` Event Type**

Emit a `"thinking"` SSE event when `on_node_start` fires for a node named `think_node`, and a `"answering"` event for `answer_node`. Update the HTML client to show these with distinct styling.

---

**Exercise 2 — Tool Result Event**

Add an `on_tool_end` handler that emits a `"tool_result"` SSE event containing the tool's output. Update the HTML client to display tool results in a different color.

---

**Exercise 3 — Multi-Turn Frontend**

Extend the HTML client to support multi-turn conversation:
1. Keep a session ID in `localStorage` so the same session persists across page reloads
2. Display the conversation history above the current stream
3. Add a "New conversation" button that generates a new session ID

---

**Exercise 4 — Streaming Word Count**

Modify the SSE server to count tokens as they stream and emit a final `"stats"` event after `"done"`: `{"tokens": 42, "latency_ms": 1230}`. Display this in the HTML client.

---
## Answer Key

Attempt the exercises before reading below.

In [ ]:
# Answer 2 — Tool Result Event (server-side addition)
# Add this to the generate() function in the FastAPI server:

SERVER_TOOL_RESULT_PATCH = '''
# In the generate() function, add this branch:
elif ename == "on_tool_end":
    output = event["data"].get("output", "")
    # output may be a ToolMessage or string — normalize
    result_str = output.content if hasattr(output, "content") else str(output)
    yield {"event": "tool_result", "data": result_str[:200]}
'''

# Frontend addition (add to the <script> block):
FRONTEND_TOOL_RESULT = '''
es.addEventListener('tool_result', e => {
  output.innerHTML += `<span class="tool-result">  Result: ${e.data}</span>\n`;
});
// In <style>:
// .tool-result { color: #009966; font-size: 0.9em; }
'''

print("Server patch:")
print(SERVER_TOOL_RESULT_PATCH)
print("\nFrontend addition:")
print(FRONTEND_TOOL_RESULT)

In [ ]:
# Answer 4 — Streaming Word Count / Stats Event
import time as _time

async def stream_with_stats(question: str):
    t0          = _time.time()
    token_count = 0

    async for event in tool_app.astream_events(
        {"question": question, "messages": []}, version="v1"
    ):
        if event["event"] == "on_chat_model_stream":
            chunk = event["data"]["chunk"].content
            if chunk:
                print(chunk, end="", flush=True)
                token_count += 1

    latency_ms = round((_time.time() - t0) * 1000)
    stats = {"tokens": token_count, "latency_ms": latency_ms}
    print(f"\n\n[stats] {stats}")

    # In a FastAPI generator, after the "done" event:
    # yield {"event": "stats", "data": json.dumps(stats)}
    return stats

stats = await stream_with_stats("What is 144 / 12?")
print(f"\nFinal stats: {stats}")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*